In [1]:
%load_ext autoreload
%autoreload 2

<h1><b>Quantization</b></h1>
<p>quantazation.ipynb</p>

<h2><b>Calibration</b></h2>

In [4]:
MODEL_NAME = "TEST"
DATASET_PATH = "./dataset/nyudepthv2"
BATCH_SIZE = 1
VITIS_AI_VERSION = 1.4 #Do not forget to change this everytime you change versions

import os
model_dir_path = os.path.join("./models/", MODEL_NAME)
build_dir_path = os.path.join(model_dir_path, f"build_{VITIS_AI_VERSION}")
if not os.path.exists(build_dir_path):
    os.makedirs(build_dir_path)
quant_model_path = os.path.join(build_dir_path, "quant_model")
if not os.path.exists(quant_model_path):
    os.makedirs(quant_model_path)

In [5]:
import torch
import src.utils

from pytorch_nndct.apis import torch_quantizer
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from src.dataset import Nyudepth_png
from src.model import UNet
from src.utils import get_dataframe

print("Starting calibration...")

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

#--- load model

model, history = src.utils.load_for_eval(UNet, MODEL_NAME, device)
model.eval()

print("\t|- Dataset :")
print("\t\t|- ", end="")
#--- generate the calibration dataloader
calibration_dataframe = get_dataframe(DATASET_PATH, "train").iloc[:200].reset_index(drop=True)
calibration_dataset = Nyudepth_png(os.path.join(DATASET_PATH, "train"), calibration_dataframe)
calibration_loader = DataLoader(
    calibration_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4
)

#--- preapare and calibrate
dummy_input = torch.randn([BATCH_SIZE, 3, 224, 224])
quantizer = torch_quantizer('calib', model, (dummy_input), output_dir=quant_model_path)
quantized_model = quantizer.quant_model

with torch.no_grad():
    for images, depth_gt in tqdm(calibration_loader, desc="Calibration"):
        _ = quantized_model(images)
print("-"*40)
print(f"Calibration complete : {quant_model_path}")
#--- export
quantizer.export_quant_config()

Starting calibration...
	|- Model :
	|	|- Loading checkpoint from './models/TEST/TEST.pth'...
	|	|- Model state restored successfully (epoch 20).
	|	|- Loading training history from './models/TEST/TEST_history.json'.
	|- Dataset :
		|- Files found : 47584 pairs rgb/depth

[VAIQ_WARN]: CUDA is not available, change device to CPU

[VAIQ_NOTE]: Quantization calibration process start up...

[VAIQ_NOTE]: =>Quant Module is in 'cpu'.

[VAIQ_NOTE]: =>Parsing UNet...

[VAIQ_NOTE]: =>Doing weights equalization...

[VAIQ_NOTE]: =>Quantizable module is generated.(./models/TEST/build_1.4/quant_model/UNet.py)

[VAIQ_NOTE]: =>Get module with quantization.


Calibration:   0%|          | 0/200 [00:00<?, ?it/s]

----------------------------------------
Calibration complete : ./models/TEST/build_1.4/quant_model

[VAIQ_NOTE]: =>Exporting quant config.(./models/TEST/build_1.4/quant_model/quant_info.json)


In [6]:
print("Starting export...")

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

#--- load float model
model, history = src.utils.load_for_eval(UNet, MODEL_NAME, device)
model.eval()

#--- generate the dataloader
print("\t|- Dataset :")
print("\t\t|- ", end="")
test_dataframe = get_dataframe(DATASET_PATH, "test")
test_dataset = Nyudepth_png(os.path.join(DATASET_PATH, "test"), test_dataframe)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=1, shuffle=False) #BATCH_SIZE Necesarly 1 for DPU

#--- load quantized model
dummy_input = torch.randn([1, 3, 224, 224])
quantizer = torch_quantizer('test', model, (dummy_input), output_dir=quant_model_path)
quantized_model = quantizer.quant_model
quantized_model.eval()

with torch.no_grad():
    images, depth_gt = next(iter(test_loader))
    _ = quantized_model(images)

quantizer.export_xmodel(deploy_check=False, output_dir=quant_model_path)
print("-"*40)
print("XMODEL export complete :", quant_model_path)

Starting export...
	|- Model :
	|	|- Loading checkpoint from './models/TEST/TEST.pth'...
	|	|- Model state restored successfully (epoch 20).
	|	|- Loading training history from './models/TEST/TEST_history.json'.
	|- Dataset :
		|- Files found : 654 pairs rgb/depth

[VAIQ_WARN]: CUDA is not available, change device to CPU

[VAIQ_NOTE]: Quantization test process start up...

[VAIQ_NOTE]: =>Quant Module is in 'cpu'.

[VAIQ_NOTE]: =>Parsing UNet...


[autoreload of nndct_UNet failed: Traceback (most recent call last):
  File "/opt/vitis_ai/conda/envs/vitis-ai-pytorch/lib/python3.6/site-packages/IPython/extensions/autoreload.py", line 245, in check
    superreload(m, reload, self.old_objects)
  File "/opt/vitis_ai/conda/envs/vitis-ai-pytorch/lib/python3.6/site-packages/IPython/extensions/autoreload.py", line 394, in superreload
    module = reload(module)
  File "/opt/vitis_ai/conda/envs/vitis-ai-pytorch/lib/python3.6/imp.py", line 315, in reload
    return importlib.reload(module)
  File "/opt/vitis_ai/conda/envs/vitis-ai-pytorch/lib/python3.6/importlib/__init__.py", line 166, in reload
    _bootstrap._exec(spec, module)
  File "<frozen importlib._bootstrap>", line 600, in _exec
AttributeError: 'NoneType' object has no attribute 'name'
]



[VAIQ_NOTE]: =>Doing weights equalization...


Exception ignored in: <bound method _MultiProcessingDataLoaderIter.__del__ of <torch.utils.data.dataloader._MultiProcessingDataLoaderIter object at 0x7f9629db8630>>
Traceback (most recent call last):
  File "/opt/vitis_ai/conda/envs/vitis-ai-pytorch/lib/python3.6/site-packages/torch/utils/data/dataloader.py", line 961, in __del__
    self._shutdown_workers()
  File "/opt/vitis_ai/conda/envs/vitis-ai-pytorch/lib/python3.6/site-packages/torch/utils/data/dataloader.py", line 941, in _shutdown_workers
    w.join()
  File "/opt/vitis_ai/conda/envs/vitis-ai-pytorch/lib/python3.6/multiprocessing/process.py", line 124, in join
    res = self._popen.wait(timeout)
  File "/opt/vitis_ai/conda/envs/vitis-ai-pytorch/lib/python3.6/multiprocessing/popen_fork.py", line 50, in wait
    return self.poll(os.WNOHANG if timeout == 0.0 else 0)
  File "/opt/vitis_ai/conda/envs/vitis-ai-pytorch/lib/python3.6/multiprocessing/popen_fork.py", line 28, in poll
    pid, sts = os.waitpid(self.pid, flag)
KeyboardInt


[VAIQ_NOTE]: =>Quantizable module is generated.(./models/TEST/build_1.4/quant_model/UNet.py)

[VAIQ_NOTE]: =>Get module with quantization.

[VAIQ_NOTE]: =>Converting to xmodel ...

[VAIQ_NOTE]: =>Successfully convert 'UNet' to xmodel.(./models/TEST/build_1.4/quant_model/UNet_int.xmodel)
----------------------------------------
XMODEL export complete : ./models/TEST/build_1.4/quant_model
